# Modeller tränas på träning och valideringsdata

In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Projektets rotmapp
PROJECT_ROOT = Path.cwd().parent

# Gör projektets src-mapp tillgänglig för importer
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importera projektets funktioner
from src.config import load_config
from src.validate_process import validate_data
from src.features import add_wind_features

# Ladda konfigurationen
cfg = load_config(
    PROJECT_ROOT / "config.toml",
    PROJECT_ROOT
)

print("Projektets rotmapp:", PROJECT_ROOT)
print("Träningsdata:", cfg.train_merged)
print("Valideringsdata:", cfg.val_merged)
print("Prognoshorisont:", cfg.horisont, "timmar")

Projektets rotmapp: c:\__Projekts__\EC_Utbildning\Projekt_PM2.5\predicting_PM2.5
Träningsdata: c:\__Projekts__\EC_Utbildning\Projekt_PM2.5\predicting_PM2.5\data\merged\train_merged_2023-05-26_2025-12-31_pm25tidsserie_6247_vaderstationid_52350.csv
Valideringsdata: c:\__Projekts__\EC_Utbildning\Projekt_PM2.5\predicting_PM2.5\data\merged\val_merged_2026-01-01_2026-04-30_pm25tidsserie_6247_vaderstationid_52350.csv
Prognoshorisont: 6 timmar


## Läser in data

In [6]:
# Läs in träningsdata
train_raw = pd.read_csv(cfg.train_merged)

# Läs in valideringsdata
val_raw = pd.read_csv(cfg.val_merged)

print("Träningsdata:", train_raw.shape)
print("Valideringsdata:", val_raw.shape)

Träningsdata: (22825, 7)
Valideringsdata: (2879, 7)


In [7]:
val_raw.tail()

,tid,pm25,Lufttemperatur,Vindriktning,Vindhastighet,Relativ luftfuktighet,Nederbördsmängd
2874,2026-04-30 19:00:00+02:00,6.22,16.2,336.0,3.6,38,0.0
2875,2026-04-30 20:00:00+02:00,7.13,13.8,332.0,3.6,49,0.0
2876,2026-04-30 21:00:00+02:00,9.15,12.3,326.0,2.4,58,0.0
2877,2026-04-30 22:00:00+02:00,8.92,11.3,322.0,1.9,63,0.0
2878,2026-04-30 23:00:00+02:00,8.76,10.5,326.0,2.3,68,0.0


In [8]:
print("Träningsdata:")
print(train_raw.columns.tolist())

print("\nValideringsdata:")
print(val_raw.columns.tolist())

Träningsdata:
['tid', 'pm25', 'Lufttemperatur', 'Vindriktning', 'Vindhastighet', 'Relativ luftfuktighet', 'Nederbördsmängd']

Valideringsdata:
['tid', 'pm25', 'Lufttemperatur', 'Vindriktning', 'Vindhastighet', 'Relativ luftfuktighet', 'Nederbördsmängd']


## Validering

In [9]:
# Validera träningsdata
train_df = validate_data(train_raw)

# Validera valideringsdata
val_df = validate_data(val_raw)

print("Träningsdata efter validering:", train_df.shape)
print("Valideringsdata efter validering:", val_df.shape)

Träningsdata efter validering: (22825, 7)
Valideringsdata efter validering: (2879, 7)


In [11]:
print("Träningsdata – saknade värden:")
print(train_df.isna().sum())

print("\nValideringsdata – saknade värden:")
print(val_df.isna().sum())

train_df.info()
val_df.info()

Träningsdata – saknade värden:
tid                         0
pm25                      154
Lufttemperatur            536
Vindriktning              572
Vindhastighet             572
Relativ luftfuktighet     570
Nederbördsmängd          1069
dtype: int64

Valideringsdata – saknade värden:
tid                       0
pm25                      3
Lufttemperatur            0
Vindriktning              2
Vindhastighet             2
Relativ luftfuktighet     0
Nederbördsmängd          48
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 22825 entries, 0 to 22824
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype                           
---  ------                 --------------  -----                           
 0   tid                    22825 non-null  datetime64[us, Europe/Stockholm]
 1   pm25                   22671 non-null  float64                         
 2   Lufttemperatur         22289 non-null  float64                         
 3   Vindriktning   

## Feature Engineering

Omvandlar vindriktning

In [12]:
# Lägg till vindriktning som sinus och cosinus
train_df = add_wind_features(train_df)
val_df = add_wind_features(val_df)

In [13]:
display(
    train_df[
        ["Vindriktning", "Vindhastighet", "vind_sin", "vind_cos"]
    ].head(10)
)

,Vindriktning,Vindhastighet,vind_sin,vind_cos
0,318.0,4.5,-0.669131,0.743145
1,308.0,4.1,-0.788011,0.615661
2,308.0,4.1,-0.788011,0.615661
3,304.0,4.6,-0.829038,0.559193
4,302.0,4.0,-0.848048,0.529919
5,306.0,4.4,-0.809017,0.587785
6,302.0,3.8,-0.848048,0.529919
7,306.0,4.1,-0.809017,0.587785
8,306.0,3.6,-0.809017,0.587785
9,318.0,5.6,-0.669131,0.743145


In [14]:
display(
    val_df[
        ["Vindriktning", "Vindhastighet", "vind_sin", "vind_cos"]
    ].head(10)
)

,Vindriktning,Vindhastighet,vind_sin,vind_cos
0,206.0,1.8,-0.438371,-0.898794
1,205.0,2.1,-0.422618,-0.906308
2,233.0,2.4,-0.798636,-0.601815
3,242.0,3.4,-0.882948,-0.469472
4,247.0,4.6,-0.920505,-0.390731
5,249.0,5.2,-0.933580,-0.358368
6,241.0,4.5,-0.874620,-0.484810
7,238.0,3.9,-0.848048,-0.529919
8,236.0,3.8,-0.829038,-0.559193
9,233.0,4.3,-0.798636,-0.601815


In [15]:
# Kontrollera observationer där vinden står stilla
vindstilla = train_df["Vindhastighet"] == 0

print("Antal vindstilla observationer:", vindstilla.sum())

display(
    train_df.loc[
        vindstilla,
        ["Vindriktning", "Vindhastighet", "vind_sin", "vind_cos"]
    ].head(10)
)

Antal vindstilla observationer: 2267


,Vindriktning,Vindhastighet,vind_sin,vind_cos
30,0.0,0.0,0.0,0.0
50,0.0,0.0,0.0,0.0
51,0.0,0.0,0.0,0.0
71,0.0,0.0,0.0,0.0
101,0.0,0.0,0.0,0.0
102,0.0,0.0,0.0,0.0
118,0.0,0.0,0.0,0.0
120,0.0,0.0,0.0,0.0
121,0.0,0.0,0.0,0.0
122,0.0,0.0,0.0,0.0


In [16]:
print("Träningsdata:")
print(train_df[["vind_sin", "vind_cos"]].isna().sum())

print("\nValideringsdata:")
print(val_df[["vind_sin", "vind_cos"]].isna().sum())

Träningsdata:
vind_sin    572
vind_cos    572
dtype: int64

Valideringsdata:
vind_sin    2
vind_cos    2
dtype: int64


## Skapar target och tar bort saknade värden

In [17]:
# Kopiera DataFrames innan vi skapar målvariabeln
train_df = train_df.copy()
val_df = val_df.copy()

# Skapa målvariabeln för PM2.5 sex timmar framåt
train_df["target"] = train_df["pm25"].shift(-cfg.horisont)
val_df["target"] = val_df["pm25"].shift(-cfg.horisont)

In [18]:
display(
    train_df[["tid", "pm25", "target"]].head(10)
)

display(
    train_df[["tid", "pm25", "target"]].tail(10)
)

,tid,pm25,target
0,2023-05-26 00:00:00+02:00,NaN,13.905
1,2023-05-26 01:00:00+02:00,NaN,13.739
2,2023-05-26 02:00:00+02:00,13.042,14.138
3,2023-05-26 03:00:00+02:00,13.116,12.972
4,2023-05-26 04:00:00+02:00,13.728,10.587
5,2023-05-26 05:00:00+02:00,13.913,10.422
6,2023-05-26 06:00:00+02:00,13.905,8.623
7,2023-05-26 07:00:00+02:00,13.739,7.389
8,2023-05-26 08:00:00+02:00,14.138,6.583
9,2023-05-26 09:00:00+02:00,12.972,6.217


,tid,pm25,target
22815,2025-12-31 14:00:00+01:00,3.962,37.966
22816,2025-12-31 15:00:00+01:00,5.169,14.523
22817,2025-12-31 16:00:00+01:00,3.025,9.805
22818,2025-12-31 17:00:00+01:00,3.458,15.261
22819,2025-12-31 18:00:00+01:00,7.554,NaN
22820,2025-12-31 19:00:00+01:00,17.473,NaN
22821,2025-12-31 20:00:00+01:00,37.966,NaN
22822,2025-12-31 21:00:00+01:00,14.523,NaN
22823,2025-12-31 22:00:00+01:00,9.805,NaN
22824,2025-12-31 23:00:00+01:00,15.261,NaN


In [19]:
display(
    val_df[["tid", "pm25", "target"]].head(10)
)


display(
    val_df[["tid", "pm25", "target"]].tail(10)
)

,tid,pm25,target
0,2026-01-01 00:00:00+01:00,NaN,4.39
1,2026-01-01 01:00:00+01:00,34.80,3.76
2,2026-01-01 02:00:00+01:00,13.50,1.89
3,2026-01-01 03:00:00+01:00,8.11,3.29
4,2026-01-01 04:00:00+01:00,8.05,3.09
5,2026-01-01 05:00:00+01:00,3.79,2.35
6,2026-01-01 06:00:00+01:00,4.39,2.30
7,2026-01-01 07:00:00+01:00,3.76,2.69
8,2026-01-01 08:00:00+01:00,1.89,2.07
9,2026-01-01 09:00:00+01:00,3.29,2.26


,tid,pm25,target
2869,2026-04-30 14:00:00+02:00,2.48,7.13
2870,2026-04-30 15:00:00+02:00,3.39,9.15
2871,2026-04-30 16:00:00+02:00,4.05,8.92
2872,2026-04-30 17:00:00+02:00,4.72,8.76
2873,2026-04-30 18:00:00+02:00,4.99,NaN
2874,2026-04-30 19:00:00+02:00,6.22,NaN
2875,2026-04-30 20:00:00+02:00,7.13,NaN
2876,2026-04-30 21:00:00+02:00,9.15,NaN
2877,2026-04-30 22:00:00+02:00,8.92,NaN
2878,2026-04-30 23:00:00+02:00,8.76,NaN


In [20]:
features = [
    "pm25",
    "Lufttemperatur",
    "vind_sin",
    "vind_cos",
    "Vindhastighet",
    "Relativ luftfuktighet",
    "Nederbördsmängd",
]

In [21]:
# Ta bort rader med saknade värden i träningsdata
train_model = train_df.dropna(
    subset=features + ["target"]
).copy()

# Ta bort rader med saknade värden i valideringsdata
val_model = val_df.dropna(
    subset=features + ["target"]
).copy()

In [22]:
print("Träningsdata före borttagning:", len(train_df))
print("Träningsdata efter borttagning:", len(train_model))

print("\nValideringsdata före borttagning:", len(val_df))
print("Valideringsdata efter borttagning:", len(val_model))

Träningsdata före borttagning: 22825
Träningsdata efter borttagning: 21555

Valideringsdata före borttagning: 2879
Valideringsdata efter borttagning: 2818


In [23]:
print("Saknade värden i träningsdata:")
print(train_model[features + ["target"]].isna().sum())

print("\nSaknade värden i valideringsdata:")
print(val_model[features + ["target"]].isna().sum())

Saknade värden i träningsdata:
pm25                     0
Lufttemperatur           0
vind_sin                 0
vind_cos                 0
Vindhastighet            0
Relativ luftfuktighet    0
Nederbördsmängd          0
target                   0
dtype: int64

Saknade värden i valideringsdata:
pm25                     0
Lufttemperatur           0
vind_sin                 0
vind_cos                 0
Vindhastighet            0
Relativ luftfuktighet    0
Nederbördsmängd          0
target                   0
dtype: int64


## Kontrollerar tidslinjen innan modellträning

In [24]:
for name, df in [
    ("Träning", train_df),
    ("Validering", val_df),
]:
    timestamps = pd.DatetimeIndex(df["tid"])

    print(f"\n{name}")
    print("Antal rader:", len(df))
    print("Dubbletter:", timestamps.duplicated().sum())
    print("Sorterad:", timestamps.is_monotonic_increasing)

    # Tidsskillnad mellan intilliggande observationer
    differences = timestamps.to_series().diff()

    print("Antal intervall som inte är en timme:",
          (differences.dropna() != pd.Timedelta(hours=1)).sum())

    print("Vanligaste tidsintervall:")
    print(differences.value_counts().head())


Träning
Antal rader: 22825
Dubbletter: 0
Sorterad: True
Antal intervall som inte är en timme: 0
Vanligaste tidsintervall:
tid
0 days 01:00:00    22824
Name: count, dtype: int64

Validering
Antal rader: 2879
Dubbletter: 0
Sorterad: True
Antal intervall som inte är en timme: 0
Vanligaste tidsintervall:
tid
0 days 01:00:00    2878
Name: count, dtype: int64


## Förbered data för modellering

In [25]:
# Features som modellerna ska använda
features = [
    "pm25",
    "Lufttemperatur",
    "vind_sin",
    "vind_cos",
    "Vindhastighet",
    "Relativ luftfuktighet",
    "Nederbördsmängd",
]

# Dela upp träningsdata
X_train = train_model[features]
y_train = train_model["target"]

# Dela upp valideringsdata
X_val = val_model[features]
y_val = val_model["target"]

In [26]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (21555, 7)
y_train: (21555,)

X_val: (2818, 7)
y_val: (2818,)


In [27]:
print("Features:")
print(X_train.columns.tolist())

print("\nSaknade värden i X_train:", X_train.isna().sum().sum())
print("Saknade värden i X_val:", X_val.isna().sum().sum())

Features:
['pm25', 'Lufttemperatur', 'vind_sin', 'vind_cos', 'Vindhastighet', 'Relativ luftfuktighet', 'Nederbördsmängd']

Saknade värden i X_train: 0
Saknade värden i X_val: 0


In [28]:
print("Träningsdata – target:")
display(y_train.describe())

print("\nValideringsdata – target:")
display(y_val.describe())

Träningsdata – target:


count    21555.000000
mean         7.582796
std          6.332880
min          0.396000
25%          3.694000
50%          5.741000
75%          9.398000
max         69.138000
Name: target, dtype: float64


Valideringsdata – target:


count    2818.000000
mean       10.523201
std         8.818755
min         1.040000
25%         4.540000
50%         7.950000
75%        12.700000
max        56.500000
Name: target, dtype: float64

## Baseline (Persistence)

Vi vill förutsäga PM2.5 sex timmar framåt.

Den enklaste referensmodellen är att anta att PM2.5 om sex timmar kommer att vara samma som det är just nu.

In [29]:
# Persistence-baseline för träningsdata
y_pred_train_baseline = X_train["pm25"]

# Persistence-baseline för valideringsdata
y_pred_val_baseline = X_val["pm25"]

In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [31]:
def evaluate_model(y_true, y_pred):
    """
    Beräknar MAE, RMSE och R² för en modell.
    """
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

In [32]:
baseline_train_results = evaluate_model(
    y_train,
    y_pred_train_baseline
)

baseline_val_results = evaluate_model(
    y_val,
    y_pred_val_baseline
)

print("Baseline – träningsdata:")
print(baseline_train_results)

print("\nBaseline – valideringsdata:")
print(baseline_val_results)

Baseline – träningsdata:
{'MAE': 2.6209822315008116, 'RMSE': 4.176965420355851, 'R2': 0.564949738727209}

Baseline – valideringsdata:
{'MAE': 3.3206493967352735, 'RMSE': 4.989168797248485, 'R2': 0.6798186584256289}


## Importerar modellerna

In [33]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Skapar våra modeller

In [34]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42,
    ),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=1.0, epsilon=0.1)),
    ]),
}

# Tränar och utvärderar Modeller

In [35]:
results = []

trained_models = {}

for name, model in models.items():
    print(f"Tränar {name}...")

    # Träna endast på träningsdata
    model.fit(X_train, y_train)

    # Prediktioner på träningsdata
    y_pred_train = model.predict(X_train)

    # Prediktioner på valideringsdata
    y_pred_val = model.predict(X_val)

    # Utvärdera båda datamängderna
    train_metrics = evaluate_model(y_train, y_pred_train)
    val_metrics = evaluate_model(y_val, y_pred_val)

    results.append({
        "Modell": name,
        "Train MAE": train_metrics["MAE"],
        "Train RMSE": train_metrics["RMSE"],
        "Train R²": train_metrics["R2"],
        "Validation MAE": val_metrics["MAE"],
        "Validation RMSE": val_metrics["RMSE"],
        "Validation R²": val_metrics["R2"],
    })

    trained_models[name] = model

print("Alla modeller är tränade.")

Tränar Linear Regression...
Tränar Ridge Regression...
Tränar Random Forest...
Tränar HistGradientBoosting...
Tränar SVR...
Alla modeller är tränade.


# Resultat

In [36]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Validation MAE"
).reset_index(drop=True)

display(results_df.round(3))

,Modell,Train MAE,Train RMSE,Train R²,Validation MAE,Validation RMSE,Validation R²
0,Linear Regression,2.516,3.920,0.617,3.210,4.772,0.707
1,Ridge Regression,2.516,3.920,0.617,3.210,4.772,0.707
2,SVR,2.342,3.781,0.644,3.220,4.812,0.702
3,HistGradientBoosting,2.136,3.185,0.747,3.270,4.892,0.692
4,Random Forest,0.845,1.320,0.957,3.497,5.132,0.661


## Jämför med baseline

In [37]:
baseline_comparison = pd.DataFrame([
    {
        "Modell": "Persistence Baseline",
        "Validation MAE": baseline_val_results["MAE"],
        "Validation RMSE": baseline_val_results["RMSE"],
        "Validation R²": baseline_val_results["R2"],
    }
])

model_comparison = results_df[
    ["Modell", "Validation MAE", "Validation RMSE", "Validation R²"]
]

comparison_df = pd.concat(
    [baseline_comparison, model_comparison],
    ignore_index=True,
)

comparison_df = comparison_df.sort_values(
    by="Validation MAE"
).reset_index(drop=True)

display(comparison_df.round(3))

,Modell,Validation MAE,Validation RMSE,Validation R²
0,Linear Regression,3.210,4.772,0.707
1,Ridge Regression,3.210,4.772,0.707
2,SVR,3.220,4.812,0.702
3,HistGradientBoosting,3.270,4.892,0.692
4,Persistence Baseline,3.321,4.989,0.680
5,Random Forest,3.497,5.132,0.661


## Sammanfattning av modellresultat

Fem maskininlärningsmodeller tränades och utvärderades för att förutsäga PM2.5-koncentrationen sex timmar framåt. Modellerna jämfördes med en persistence baseline, som använder det aktuella PM2.5-värdet som prognos för sex timmar senare.

**Linear Regression och Ridge Regression presterade bäst** på valideringsdata, med MAE på 3,210, RMSE på 4,772 och R² på 0,707. SVR presterade nästan lika bra, medan HistGradientBoosting hade något sämre resultat. Random Forest presterade sämst på valideringsdata, trots mycket goda resultat på träningsdata, vilket tyder på överanpassning.

Jämfört med persistence baseline, vars MAE var 3,321, minskade Linear Regression och Ridge Regression MAE med cirka 3,3 %. Detta innebär att maskininlärningsmodellerna förbättrade prognoserna, men att förbättringen var relativt begränsad.

Resultaten visar att enklare linjära modeller för närvarande fungerar bäst med de valda variablerna. Linear Regression väljs därför som preliminär huvudmodell, medan Ridge Regression behålls som jämförelsemodell.

**Nästa steg** är att analysera modellernas koefficienter och residualer samt visualisera förutsägelserna. Därefter kan eventuell modelloptimering genomföras. Testdata ska förbli orörd fram till den slutliga utvärderingen, så att den kan användas för en så rättvisande bedömning som möjligt av modellens generaliseringsförmåga.

# #Experminet : för feature selection

- Aktuell PM2.5 endast den aktuell koncentrationen

In [38]:
features_pm25 = ["pm25"]


features_pm25 = ["pm25"]

X_train_pm25 = train_model[features_pm25]
y_train_pm25 = train_model["target"]

X_val_pm25 = val_model[features_pm25]
y_val_pm25 = val_model["target"]

In [39]:
from sklearn.linear_model import LinearRegression

model_pm25 = LinearRegression()

model_pm25.fit(X_train_pm25, y_train_pm25)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[0.78]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['pm25']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.652
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


In [40]:
y_pred_train_pm25 = model_pm25.predict(X_train_pm25)
y_pred_val_pm25 = model_pm25.predict(X_val_pm25)

train_pm25_results = evaluate_model(
    y_train_pm25,
    y_pred_train_pm25,
)

val_pm25_results = evaluate_model(
    y_val_pm25,
    y_pred_val_pm25,
)

In [41]:
pm25_results_df = pd.DataFrame([
    {
        "Dataset": "Träning",
        **train_pm25_results,
    },
    {
        "Dataset": "Validering",
        **val_pm25_results,
    },
])

display(pm25_results_df.round(3))

,Dataset,MAE,RMSE,R2
0,Träning,2.524,3.948,0.611
1,Validering,3.227,4.842,0.698


In [43]:
pm25_comparison = pd.DataFrame([
    {
        "Modell": "Linear Regression – alla features",
        "MAE": 3.210,
        "RMSE": 4.772,
        "R2": 0.707,
    },
    {
        "Modell": "Linear Regression – endast PM2.5",
        "MAE": val_pm25_results["MAE"],
        "RMSE": val_pm25_results["RMSE"],
        "R2": val_pm25_results["R2"],
    },
])


pm25_comparison.sort_values("MAE").reset_index(drop=True).round(3)


,Modell,MAE,RMSE,R2
0,Linear Regression – alla features,3.210,4.772,0.707
1,Linear Regression – endast PM2.5,3.227,4.842,0.698
